# Modeling Human Activity States Using Hidden Markov Models

## Background and Motivation

Human Activity Recognition (HAR) is a fundamental challenge in ubiquitous computing, enabling applications in healthcare monitoring, smart environments, and fitness tracking. In this project, we use smartphone inertial sensors — accelerometer and gyroscope — to distinguish between two activities: **Standing** and **Still (no movement)**. While these activities appear superficially similar, subtle differences in sensor patterns (e.g., micro-vibrations from breathing and body sway during standing versus near-zero motion when a phone lies flat) make them distinguishable. Hidden Markov Models (HMMs) are well-suited for this task because they explicitly model the temporal dynamics of sequential sensor data, allowing us to infer latent activity states from noisy observations.

---

## 0. Setup and Imports

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import signal
from scipy.fft import fft, fftfreq
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report
from hmmlearn.hmm import GaussianHMM
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)

print('All imports successful!')

---
## 1. Data Collection & Preprocessing

### 1.1 Extract Data from Zip Files

Update the two folder paths below to point to your zip file directories.

In [ ]:
# ============================================================
 # folder containing standing zip files
datacollect =   "path/to/standing/zips" 
OUTPUT_DIR    = "sensor_data"             # Extracted CSVs 
# ============================================================

NEEDED_FILES = ["Accelerometer.csv", "Gyroscope.csv"]

def extract_zips(folders: dict, output_dir: str):
    """Extract only Accelerometer.csv and Gyroscope.csv from each zip."""
    os.makedirs(output_dir, exist_ok=True)
    for activity, zip_folder in folders.items():
        print(f"\nProcessing '{activity}' recordings...")
        zips = sorted([f for f in os.listdir(zip_folder) if f.endswith(".zip")])
        for i, zip_name in enumerate(zips, start=1):
            zip_path = os.path.join(zip_folder, zip_name)
            extract_path = os.path.join(output_dir, f"{activity}_{i:02d}")
            os.makedirs(extract_path, exist_ok=True)
            with zipfile.ZipFile(zip_path, 'r') as z:
                for file in z.namelist():
                    if os.path.basename(file) in NEEDED_FILES:
                        data = z.read(file)
                        out_path = os.path.join(extract_path, os.path.basename(file))
                        with open(out_path, 'wb') as f:
                            f.write(data)
            print(f"  Extracted: {activity}_{i:02d}")
    print("\nExtraction complete!")

folders = {"standing": STANDING_ZIPS, "still": STILL_ZIPS}
extract_zips(folders, OUTPUT_DIR)

### 1.2 Load and Merge Sensor Files

In [ ]:
def load_recording(folder_path: str) -> pd.DataFrame:
    """
    Load and merge Accelerometer.csv and Gyroscope.csv from a recording folder.
    Merges on nearest timestamp to handle slight timing differences.
    """
    acc_path  = os.path.join(folder_path, "Accelerometer.csv")
    gyro_path = os.path.join(folder_path, "Gyroscope.csv")

    acc  = pd.read_csv(acc_path)
    gyro = pd.read_csv(gyro_path)

    # Sensor Logger uses 'time' column (nanoseconds or seconds)
    # Normalize column names to lowercase
    acc.columns  = [c.lower() for c in acc.columns]
    gyro.columns = [c.lower() for c in gyro.columns]

    # Rename sensor axes to avoid collision after merge
    acc  = acc.rename(columns={'x': 'acc_x', 'y': 'acc_y', 'z': 'acc_z'})
    gyro = gyro.rename(columns={'x': 'gyro_x', 'y': 'gyro_y', 'z': 'gyro_z'})

    # Sort by time
    acc  = acc.sort_values('time').reset_index(drop=True)
    gyro = gyro.sort_values('time').reset_index(drop=True)

    # Merge on nearest timestamp
    merged = pd.merge_asof(acc[['time','acc_x','acc_y','acc_z']],
                           gyro[['time','gyro_x','gyro_y','gyro_z']],
                           on='time', direction='nearest')
    merged = merged.dropna().reset_index(drop=True)
    return merged


def load_all_recordings(data_dir: str):
    """
    Load all recordings from sensor_data folder.
    Returns a dict: {folder_name: dataframe}
    """
    recordings = {}
    for folder_name in sorted(os.listdir(data_dir)):
        folder_path = os.path.join(data_dir, folder_name)
        if os.path.isdir(folder_path):
            try:
                df = load_recording(folder_path)
                recordings[folder_name] = df
                print(f"Loaded: {folder_name:20s} | {len(df):5d} samples")
            except Exception as e:
                print(f"Skipped {folder_name}: {e}")
    return recordings


recordings = load_all_recordings(OUTPUT_DIR)
print(f"\nTotal recordings loaded: {len(recordings)}")

### 1.3 Inspect Sampling Rate

In [ ]:
def compute_sampling_rate(df: pd.DataFrame) -> float:
    """Estimate sampling rate in Hz from timestamp column."""
    times = df['time'].values
    # Sensor Logger timestamps may be in nanoseconds — convert if needed
    if times[0] > 1e10:   # nanoseconds
        times = times / 1e9
    diffs = np.diff(times)
    avg_interval = np.mean(diffs)
    return round(1.0 / avg_interval, 2)

print("Sampling rates per recording:")
print("-" * 40)
rates = {}
for name, df in recordings.items():
    sr = compute_sampling_rate(df)
    rates[name] = sr
    print(f"  {name:20s}: {sr} Hz")

# Use the most common sampling rate
all_rates = list(rates.values())
TARGET_SR = round(np.median(all_rates))
print(f"\nMedian sampling rate: {TARGET_SR} Hz  (used as TARGET_SR)")

### 1.4 Visualize Raw Sensor Data

In [ ]:
def plot_raw_sample(recordings: dict, activity: str, index: int = 1):
    """Plot raw accelerometer and gyroscope signals for one recording."""
    key = f"{activity}_{index:02d}"
    if key not in recordings:
        print(f"Recording {key} not found.")
        return
    df = recordings[key].copy()

    # Normalize time to start from 0
    t = df['time'].values
    if t[0] > 1e10:
        t = t / 1e9
    t = t - t[0]

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    fig.suptitle(f"Raw Sensor Data — {key.upper()}", fontsize=14, fontweight='bold')

    # Accelerometer
    axes[0].plot(t, df['acc_x'], label='acc_x', color='r', alpha=0.8)
    axes[0].plot(t, df['acc_y'], label='acc_y', color='g', alpha=0.8)
    axes[0].plot(t, df['acc_z'], label='acc_z', color='b', alpha=0.8)
    axes[0].set_ylabel('Acceleration (m/s²)')
    axes[0].legend(loc='upper right')
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title('Accelerometer')

    # Gyroscope
    axes[1].plot(t, df['gyro_x'], label='gyro_x', color='r', alpha=0.8)
    axes[1].plot(t, df['gyro_y'], label='gyro_y', color='g', alpha=0.8)
    axes[1].plot(t, df['gyro_z'], label='gyro_z', color='b', alpha=0.8)
    axes[1].set_ylabel('Angular Velocity (rad/s)')
    axes[1].set_xlabel('Time (seconds)')
    axes[1].legend(loc='upper right')
    axes[1].grid(True, alpha=0.3)
    axes[1].set_title('Gyroscope')

    plt.tight_layout()
    plt.savefig(f"raw_{key}.png", dpi=150, bbox_inches='tight')
    plt.show()

# Plot one sample per activity
plot_raw_sample(recordings, 'standing', 1)
plot_raw_sample(recordings, 'still', 1)

---
## 2. Feature Extraction (Time-Domain + Frequency-Domain)

### Window Parameters

We use a sliding window approach. **Window size** is based on the sampling rate:
- At 50 Hz, a 2-second window = 100 samples → captures enough signal for FFT while keeping temporal resolution.
- 50% overlap (step = window_size // 2) increases the number of windows without data loss.

In [ ]:
# Window parameters — based on sampling rate
WINDOW_SECONDS = 2          # 2-second window
WINDOW_SIZE    = int(TARGET_SR * WINDOW_SECONDS)   # e.g. 50Hz * 2s = 100 samples
STEP_SIZE      = WINDOW_SIZE // 2                  # 50% overlap

print(f"Window size : {WINDOW_SIZE} samples ({WINDOW_SECONDS}s at {TARGET_SR}Hz)")
print(f"Step size   : {STEP_SIZE} samples (50% overlap)")
print()
print("Justification: A 2-second window at 50Hz provides 100 samples — sufficient")
print("for FFT frequency resolution (0.5 Hz bins) while keeping temporal granularity.")
print("50% overlap maximises the number of windows from short (5-10s) recordings.")

In [ ]:
SENSOR_COLS = ['acc_x', 'acc_y', 'acc_z', 'gyro_x', 'gyro_y', 'gyro_z']

def compute_features(window: np.ndarray, fs: float) -> np.ndarray:
    """
    Compute time-domain and frequency-domain features for a single window.
    
    Parameters:
        window : (N, 6) array — [acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z]
        fs     : sampling frequency in Hz

    Returns:
        feature_vector : 1D numpy array
    """
    features = []
    N = window.shape[0]

    acc  = window[:, :3]   # acc_x, acc_y, acc_z
    gyro = window[:, 3:]   # gyro_x, gyro_y, gyro_z

    # ── TIME-DOMAIN FEATURES ──────────────────────────────────────────────

    # 1. Mean per axis
    features.extend(np.mean(window, axis=0).tolist())

    # 2. Variance per axis (captures signal spread — key for standing vs still)
    features.extend(np.var(window, axis=0).tolist())

    # 3. Standard Deviation per axis
    features.extend(np.std(window, axis=0).tolist())

    # 4. RMS (Root Mean Square) per axis — energy of signal
    rms = np.sqrt(np.mean(window**2, axis=0))
    features.extend(rms.tolist())

    # 5. Signal Magnitude Area (SMA) for acc and gyro
    sma_acc  = np.sum(np.abs(acc))  / N
    sma_gyro = np.sum(np.abs(gyro)) / N
    features.extend([sma_acc, sma_gyro])

    # 6. Correlation between acc axes (captures motion coupling)
    corr_xy = np.corrcoef(acc[:, 0], acc[:, 1])[0, 1]
    corr_xz = np.corrcoef(acc[:, 0], acc[:, 2])[0, 1]
    corr_yz = np.corrcoef(acc[:, 1], acc[:, 2])[0, 1]
    features.extend([corr_xy, corr_xz, corr_yz])

    # ── FREQUENCY-DOMAIN FEATURES (via FFT) ──────────────────────────────

    freqs = fftfreq(N, d=1.0/fs)
    pos_mask = freqs > 0

    for col_idx in range(6):
        col = window[:, col_idx]
        fft_vals   = np.abs(fft(col))
        fft_pos    = fft_vals[pos_mask]
        freqs_pos  = freqs[pos_mask]

        # 7. Dominant frequency — frequency with highest power
        dom_freq = freqs_pos[np.argmax(fft_pos)]
        features.append(dom_freq)

        # 8. Spectral energy — total power in frequency domain
        spectral_energy = np.sum(fft_pos**2) / N
        features.append(spectral_energy)

        # 9. Spectral entropy — spread of energy across frequencies
        psd = fft_pos**2
        psd_norm = psd / (np.sum(psd) + 1e-12)
        spectral_entropy = -np.sum(psd_norm * np.log(psd_norm + 1e-12))
        features.append(spectral_entropy)

    return np.array(features, dtype=np.float32)


def extract_windows(df: pd.DataFrame, window_size: int, step: int, fs: float) -> np.ndarray:
    """Slide a window over a recording and extract features from each window."""
    data = df[SENSOR_COLS].values
    feature_list = []
    for start in range(0, len(data) - window_size + 1, step):
        window = data[start : start + window_size]
        feat   = compute_features(window, fs)
        feature_list.append(feat)
    return np.array(feature_list)


# Test on one recording
sample_key = list(recordings.keys())[0]
sample_feats = extract_windows(recordings[sample_key], WINDOW_SIZE, STEP_SIZE, TARGET_SR)
print(f"Feature vector length : {sample_feats.shape[1]}")
print(f"Windows from '{sample_key}': {sample_feats.shape[0]}")

### 2.1 Build Feature Matrix for All Recordings

In [ ]:
LABEL_MAP = {'standing': 0, 'still': 1}
LABEL_NAMES = ['Standing', 'Still']

def build_dataset(recordings: dict, window_size: int, step: int, fs: float):
    """
    Build feature matrix X and label array y from all recordings.
    Also returns lengths list for HMM (needed to handle variable-length sequences).
    """
    X_list, y_list, lengths = [], [], []

    for name, df in sorted(recordings.items()):
        activity = name.rsplit('_', 1)[0]   # 'standing_01' -> 'standing'
        if activity not in LABEL_MAP:
            continue
        label  = LABEL_MAP[activity]
        feats  = extract_windows(df, window_size, step, fs)
        if len(feats) == 0:
            continue
        X_list.append(feats)
        y_list.extend([label] * len(feats))
        lengths.append(len(feats))

    X = np.vstack(X_list)
    y = np.array(y_list)
    return X, y, lengths


X_all, y_all, lengths_all = build_dataset(recordings, WINDOW_SIZE, STEP_SIZE, TARGET_SR)

print(f"Total feature matrix shape : {X_all.shape}")
print(f"Total labels               : {y_all.shape}")
print(f"Class distribution:")
for label, name in enumerate(LABEL_NAMES):
    print(f"  {name:10s}: {np.sum(y_all == label):4d} windows")

### 2.2 Feature Normalization (Z-Score)

Z-score normalization ensures each feature contributes equally regardless of its scale, preventing features with large magnitudes (e.g., spectral energy) from dominating the HMM emission probabilities.

In [ ]:
# Split into train (first ~80% of recordings) and test (last ~20%)
# We split by recordings, not windows, to avoid data leakage
def train_test_split_recordings(recordings: dict, test_ratio: float = 0.2):
    """Split recordings into train and test sets per activity."""
    train_recs, test_recs = {}, {}
    
    for activity in ['standing', 'still']:
        keys = sorted([k for k in recordings if k.startswith(activity)])
        n_test = max(1, int(len(keys) * test_ratio))
        train_keys = keys[:-n_test]
        test_keys  = keys[-n_test:]
        for k in train_keys: train_recs[k] = recordings[k]
        for k in test_keys:  test_recs[k]  = recordings[k]
        print(f"{activity}: {len(train_keys)} train, {len(test_keys)} test recordings")
    
    return train_recs, test_recs


train_recs, test_recs = train_test_split_recordings(recordings, test_ratio=0.2)

# Build train and test datasets
X_train, y_train, lengths_train = build_dataset(train_recs, WINDOW_SIZE, STEP_SIZE, TARGET_SR)
X_test,  y_test,  lengths_test  = build_dataset(test_recs,  WINDOW_SIZE, STEP_SIZE, TARGET_SR)

# Fit scaler on TRAIN only — apply to both
scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print(f"\nTrain set: {X_train.shape}")
print(f"Test set : {X_test.shape}")

---
## 3. HMM Model Definition

| Component | Description |
|---|---|
| **Hidden States (Z)** | 2 states: Standing (0), Still (1) |
| **Observations (X)** | Normalized feature vectors from sensor windows |
| **Transition Matrix (A)** | P(next state \| current state) |
| **Emission (B)** | Gaussian distributions over feature vectors per state |
| **Initial Probs (π)** | P(starting in each state) |

---
## 4. Model Implementation: Baum-Welch Training + Viterbi Decoding

In [ ]:
# ── TRAIN A SEPARATE HMM PER ACTIVITY ────────────────────────────────────
# This is the generative approach: each class has its own HMM.
# Classification = assign to the HMM that gives highest log-likelihood.

N_COMPONENTS = 2   # number of hidden states per activity HMM
N_ITER       = 200 # max Baum-Welch iterations
TOL          = 1e-4  # convergence threshold (log-likelihood delta)

hmm_models = {}

for label, activity in enumerate(LABEL_NAMES):
    # Extract windows belonging to this activity only
    mask = y_train == label
    X_act = X_train[mask]

    # Lengths for this activity (each recording = one sequence)
    activity_key = activity.lower()
    act_recs = {k: v for k, v in train_recs.items() if k.startswith(activity_key)}
    _, _, act_lengths = build_dataset(act_recs, WINDOW_SIZE, STEP_SIZE, TARGET_SR)
    # Re-scale lengths using normalized data

    model = GaussianHMM(
        n_components=N_COMPONENTS,
        covariance_type='diag',    # diagonal covariance: faster, less overfitting
        n_iter=N_ITER,
        tol=TOL,                   # convergence check: stop when log-likelihood improves < TOL
        random_state=42,
        verbose=False
    )

    # Baum-Welch training
    model.fit(X_act, lengths=act_lengths)

    hmm_models[activity] = model
    print(f"Trained HMM for '{activity}' | converged={model.monitor_.converged} | "
          f"iterations={len(model.monitor_.history)}")

print("\nAll models trained!")

In [ ]:
# ── CLASSIFICATION VIA LOG-LIKELIHOOD ────────────────────────────────────

def predict_activity(X_window: np.ndarray, models: dict) -> str:
    """
    Predict activity for a single feature window by comparing
    log-likelihoods across all trained HMMs.
    """
    scores = {}
    for activity, model in models.items():
        try:
            scores[activity] = model.score(X_window.reshape(1, -1))
        except:
            scores[activity] = -np.inf
    return max(scores, key=scores.get)


def predict_sequence(X_seq: np.ndarray, models: dict, label_map: dict) -> np.ndarray:
    """Predict labels for all windows in X_seq."""
    inv_map = {v: k.capitalize() for k, v in label_map.items()}
    preds = []
    for i in range(len(X_seq)):
        pred = predict_activity(X_seq[i], models)
        preds.append(LABEL_MAP[pred.lower()])
    return np.array(preds)


# ── VITERBI DECODING (per activity model) ────────────────────────────────

def viterbi_decode_sequence(X_seq: np.ndarray, model: GaussianHMM) -> tuple:
    """
    Use Viterbi algorithm to decode the most likely hidden state sequence.
    Returns (log_prob, state_sequence).
    """
    log_prob, state_seq = model.decode(X_seq, algorithm='viterbi')
    return log_prob, state_seq


# Demo: Viterbi decode on first test recording
print("Viterbi decoding demo:")
for label, activity in enumerate(LABEL_NAMES):
    act_recs_test = {k: v for k, v in test_recs.items() if k.startswith(activity.lower())}
    if not act_recs_test:
        continue
    first_key = list(act_recs_test.keys())[0]
    df_demo   = act_recs_test[first_key]
    X_demo    = extract_windows(df_demo, WINDOW_SIZE, STEP_SIZE, TARGET_SR)
    X_demo    = scaler.transform(X_demo)
    log_p, states = viterbi_decode_sequence(X_demo, hmm_models[activity])
    print(f"  {activity:10s} | log-prob: {log_p:.2f} | hidden states: {states}")

---
## 5. Visualizations

### 5.1 Transition Matrix Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Transition Probability Matrices', fontsize=14, fontweight='bold')

for ax, (activity, model) in zip(axes, hmm_models.items()):
    trans = model.transmat_
    state_labels = [f'State {i}' for i in range(model.n_components)]
    sns.heatmap(trans, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=state_labels, yticklabels=state_labels,
                ax=ax, vmin=0, vmax=1, linewidths=0.5)
    ax.set_title(f'{activity} HMM', fontsize=12)
    ax.set_xlabel('To State')
    ax.set_ylabel('From State')

plt.tight_layout()
plt.savefig('transition_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nInterpretation: High diagonal values indicate the model tends to stay")
print("in the same hidden state, which is expected for steady activities.")

### 5.2 Emission Probabilities (Gaussian Means per State)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Emission Probability — Gaussian Means per Hidden State', fontsize=13, fontweight='bold')

# Show first 20 features only (for readability)
N_SHOW = min(20, X_train.shape[1])

for ax, (activity, model) in zip(axes, hmm_models.items()):
    means = model.means_[:, :N_SHOW]   # (n_components, N_SHOW)
    im = ax.imshow(means, aspect='auto', cmap='coolwarm')
    plt.colorbar(im, ax=ax)
    ax.set_title(f'{activity} HMM')
    ax.set_xlabel('Feature Index (first 20)')
    ax.set_ylabel('Hidden State')
    ax.set_yticks(range(model.n_components))
    ax.set_yticklabels([f'State {i}' for i in range(model.n_components)])

plt.tight_layout()
plt.savefig('emission_probabilities.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Decoded State Sequence Visualization

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)
fig.suptitle('Viterbi Decoded Hidden State Sequences', fontsize=13, fontweight='bold')

colors = ['#2196F3', '#FF9800']

for ax, (label, activity) in enumerate(zip(range(2), LABEL_NAMES)):
    act_recs_test = {k: v for k, v in test_recs.items() if k.startswith(activity.lower())}
    if not act_recs_test:
        continue
    first_key = list(act_recs_test.keys())[0]
    X_demo    = extract_windows(act_recs_test[first_key], WINDOW_SIZE, STEP_SIZE, TARGET_SR)
    X_demo    = scaler.transform(X_demo)
    _, states = viterbi_decode_sequence(X_demo, hmm_models[activity])

    axes[ax].step(range(len(states)), states, where='post',
                  color=colors[label], linewidth=2)
    axes[ax].fill_between(range(len(states)), states, step='post',
                          alpha=0.3, color=colors[label])
    axes[ax].set_title(f'True Activity: {activity}', fontsize=11)
    axes[ax].set_xlabel('Window Index')
    axes[ax].set_ylabel('Hidden State')
    axes[ax].set_yticks([0, 1])
    axes[ax].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('decoded_sequences.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Model Evaluation on Unseen Test Data

In [ ]:
# Predict all test windows
y_pred = predict_sequence(X_test, hmm_models, LABEL_MAP)

# ── CONFUSION MATRIX ─────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES,
            linewidths=0.5, ax=ax)
ax.set_title('Confusion Matrix — Test Set', fontsize=13, fontweight='bold')
ax.set_xlabel('Predicted Label')
ax.set_ylabel('True Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── PER-CLASS METRICS ─────────────────────────────────────────────────────

def compute_metrics(cm: np.ndarray, label_names: list) -> pd.DataFrame:
    """Compute sensitivity, specificity, and accuracy per class."""
    rows = []
    total = cm.sum()
    overall_acc = np.trace(cm) / total

    for i, name in enumerate(label_names):
        TP = cm[i, i]
        FN = cm[i, :].sum() - TP
        FP = cm[:, i].sum() - TP
        TN = total - TP - FN - FP

        sensitivity = TP / (TP + FN) if (TP + FN) > 0 else 0.0   # recall
        specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
        accuracy    = (TP + TN) / total

        rows.append({
            'Activity':         name,
            'N Samples':        cm[i, :].sum(),
            'Sensitivity':      round(sensitivity, 4),
            'Specificity':      round(specificity, 4),
            'Accuracy':         round(accuracy, 4),
        })

    df_metrics = pd.DataFrame(rows)
    df_metrics.loc[len(df_metrics)] = {
        'Activity': 'OVERALL',
        'N Samples': total,
        'Sensitivity': round(overall_acc, 4),
        'Specificity': '-',
        'Accuracy': round(overall_acc, 4)
    }
    return df_metrics


metrics_df = compute_metrics(cm, LABEL_NAMES)
print("=" * 65)
print("EVALUATION RESULTS ON UNSEEN TEST DATA")
print("=" * 65)
print(metrics_df.to_string(index=False))
print()
print(classification_report(y_test, y_pred, target_names=LABEL_NAMES))

In [ ]:
# ── METRICS BAR CHART ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))

metric_rows = metrics_df[metrics_df['Activity'] != 'OVERALL'].copy()
x = np.arange(len(metric_rows))
width = 0.25

ax.bar(x - width, metric_rows['Sensitivity'].astype(float), width, label='Sensitivity', color='#2196F3')
ax.bar(x,         metric_rows['Specificity'].astype(float), width, label='Specificity', color='#4CAF50')
ax.bar(x + width, metric_rows['Accuracy'].astype(float),    width, label='Accuracy',    color='#FF9800')

ax.set_xticks(x)
ax.set_xticklabels(metric_rows['Activity'], fontsize=11)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score')
ax.set_title('Per-Activity Evaluation Metrics', fontsize=13, fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
ax.axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

# Annotate bars
for bar in ax.patches:
    ax.annotate(f'{bar.get_height():.2f}',
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3), textcoords='offset points',
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('metrics_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Analysis and Reflection

### 6.1 Initial State Probabilities

In [ ]:
print("Initial State Probabilities (π):")
print("-" * 40)
for activity, model in hmm_models.items():
    print(f"\n{activity}:")
    for i, p in enumerate(model.startprob_):
        print(f"  State {i}: {p:.4f}")

In [ ]:
print("""
==========================================================
 ANALYSIS AND REFLECTION
==========================================================

1. WHICH ACTIVITIES WERE EASIEST/HARDEST TO DISTINGUISH
   Since we have only two activities (Standing and Still),
   the challenge is subtle. Standing involves micro-vibrations
   from breathing and body sway, producing slightly higher
   variance in accelerometer data. Still (phone on flat surface)
   shows near-zero gyroscope readings and very low variance.
   The dominant frequency feature from FFT helps capture this.

2. TRANSITION PROBABILITIES
   High diagonal values in both transition matrices reflect
   the real-world behavior: during a recording session,
   the activity remains stable (you stay standing or keep
   the phone still), so self-transitions dominate.

3. EFFECT OF SENSOR NOISE / SAMPLING RATE
   At lower sampling rates, FFT frequency resolution decreases
   (fewer frequency bins), potentially merging distinct peaks.
   Calibrated sensor data from Sensor Logger reduces bias
   from gravity and sensor offset, improving feature quality.
   Harmonizing sampling rates across devices ensures consistent
   windowing and feature extraction.

4. POSSIBLE IMPROVEMENTS
   - Collect more recordings (>50 total) for better HMM training
   - Add more activities (walking, jumping) for richer distinctions
   - Try GMM-HMM with full covariance matrices
   - Use cross-validation across participants
   - Explore deep learning alternatives (LSTM) for comparison
==========================================================
""")

### 6.2 Group Metadata Table

In [ ]:
# UPDATE THIS TABLE WITH YOUR ACTUAL INFO
group_info = pd.DataFrame([
    {
        'Member':          'Member 1 (Your Name)',
        'Phone':           'e.g. Samsung Galaxy S21',
        'OS':              'Android 12',
        'Sampling Rate':   '50 Hz',
        'Activities':      'Standing, Still'
    },
    {
        'Member':          'Member 2 (Partner Name)',
        'Phone':           'e.g. iPhone 13',
        'OS':              'iOS 16',
        'Sampling Rate':   '50 Hz',
        'Activities':      'Standing, Still'
    }
])

print("Group Data Collection Summary:")
print(group_info.to_string(index=False))

### 6.3 Unseen Data Description

In [ ]:
print(f"""
UNSEEN TEST DATA DESCRIPTION
==============================
Test recordings were withheld from training by splitting per activity:
  - The last 20% of recordings per activity were reserved for testing.
  - These recordings were collected in the same session but NOT used
    during model fitting (Baum-Welch training).
  - The scaler (Z-score normalization) was fit on training data only
    and applied to test data — preventing data leakage.

For a stronger evaluation, test data could come from:
  - A different recording session (different time/day)
  - A different participant
  - A different environment (different floor, outdoor)

Total test windows evaluated : {len(y_test)}
  - Standing windows : {np.sum(y_test == 0)}
  - Still windows    : {np.sum(y_test == 1)}
""")

---
## Summary of All Output Files

| File | Description |
|---|---|
| `raw_standing_01.png` | Raw sensor signals — standing |
| `raw_still_01.png` | Raw sensor signals — still |
| `transition_matrices.png` | Transition probability heatmaps |
| `emission_probabilities.png` | Gaussian mean emission heatmaps |
| `decoded_sequences.png` | Viterbi decoded hidden state plots |
| `confusion_matrix.png` | Confusion matrix on test data |
| `metrics_chart.png` | Per-activity sensitivity/specificity/accuracy |